**Checking if data is loading from s3 bucket to databricks.**

In [0]:
%fs ls s3a://my-retail-lakehouse/bronze/

**Implementing bronze layer which stores data in a raw form without any transformations. Now with the data extracted from external sources we can transform data in next layer.**

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS bronze_db;


CREATE TABLE IF NOT EXISTS bronze_db.customers
USING csv
OPTIONS (
  path "s3a://my-retail-lakehouse/bronze/customers/",
  header "true"
);
SELECT * FROM bronze_db.customers LIMIT 10;

CREATE TABLE IF NOT EXISTS bronze_db.products
USING csv
OPTIONS (
  path "s3a://my-retail-lakehouse/bronze/products/",
  header "true"
);


CREATE TABLE IF NOT EXISTS bronze_db.stores
USING csv
OPTIONS (
  path "s3a://my-retail-lakehouse/bronze/stores/",
  header "true"
);


CREATE TABLE IF NOT EXISTS bronze_db.sales
USING csv
OPTIONS (
  path "s3a://my-retail-lakehouse/bronze/sales/",
  header "true"
);


/*Verify the data extraction*/
SELECT * FROM bronze_db.products LIMIT 5;
SELECT * FROM bronze_db.stores LIMIT 5;
SELECT * FROM bronze_db.sales LIMIT 5;

**Implementing silver layer for data cleaning and data transformation**

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS silver_db;

-- CUSTOMERS

CREATE OR REPLACE TABLE silver_db.customers AS
SELECT DISTINCT
    CustomerID,
    INITCAP(TRIM(CustomerName)) AS CustomerName,
    LOWER(TRIM(Email)) AS Email,
    TRIM(City) AS City,
    TRIM(Address) AS Address,
    to_date(LastUpdated, 'dd-MM-yyyy') AS LastUpdated
FROM bronze_db.customers
WHERE CustomerID IS NOT NULL;

-- PRODUCTS

CREATE OR REPLACE TABLE silver_db.products AS
SELECT
    ProductID,
    TRIM(ProductName) AS ProductName,
    TRIM(Category) AS Category,
    UnitPrice
FROM bronze_db.products
WHERE UnitPrice > 0;

-- STORES

CREATE OR REPLACE TABLE silver_db.stores AS
SELECT
    StoreID,
    INITCAP(TRIM(StoreName)) AS StoreName,
    TRIM(Region) AS Region
FROM bronze_db.stores
WHERE Region IS NOT NULL;

-- SALES

CREATE OR REPLACE TABLE silver_db.sales AS
SELECT DISTINCT
    TransactionID,
    CustomerID,
    ProductID,
    StoreID,
    Quantity,
    to_date(TxnDate, 'dd-MM-yyyy') AS TxnDate
FROM bronze_db.sales
WHERE Quantity > 0;

select * from silver_db.sales limit 10;

**Validating the data tranformation in the silver layer**

In [0]:
%sql
--1. CUSTOMER VALIDATION

--Check if Email is lowercase
SELECT * 
FROM silver_db.customers
WHERE Email != LOWER(Email);

--Check for leading/trailing spaces in City
SELECT * 
FROM silver_db.customers
WHERE City LIKE ' %' OR City LIKE '% ';

--Check duplicate CustomerID
SELECT CustomerID, COUNT(*) 
FROM silver_db.customers
GROUP BY CustomerID
HAVING COUNT(*) > 1;

--Check NULL CustomerID
SELECT *
FROM silver_db.customers
WHERE CustomerID IS NULL;


--2. PRODUCTS VALIDATION

--Check UnitPrice > 0
SELECT *
FROM silver_db.products
WHERE UnitPrice <= 0;

--Check trimming of ProductName
SELECT *
FROM silver_db.products
WHERE ProductName LIKE ' %' OR ProductName LIKE '% ';


--3. STORES VALIDATION

--Check NULL Region
SELECT *
FROM silver_db.stores
WHERE Region IS NULL;

--Check trimming of StoreName
SELECT *
FROM silver_db.stores
WHERE StoreName LIKE ' %' OR StoreName LIKE '% ';

--4. SALES VALIDATION

--Check Quantity > 0
SELECT *
FROM silver_db.sales
WHERE Quantity <= 0;

--Check duplicate TransactionID
SELECT TransactionID, COUNT(*)
FROM silver_db.sales
GROUP BY TransactionID
HAVING COUNT(*) > 1;


--5. REFERENTIAL INTEGRITY

--Check CustomerID in sales exists in customers
SELECT *
FROM silver_db.sales s
LEFT JOIN silver_db.customers c
ON s.CustomerID = c.CustomerID
WHERE c.CustomerID IS NULL;


--6. DATE VALIDATION

--Check for invalid dates
SELECT *
FROM silver_db.sales
WHERE TxnDate IS NULL;

**Implementing gold layer for bussiness logic and scd type 2 implementation**

**Dimension customer**

In [0]:
%sql
CREATE TABLE gold_db.dim_customer AS
SELECT
    monotonically_increasing_id() AS CustomerSK,
    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    CURRENT_DATE() AS StartDate,
    DATE '9999-12-31' AS EndDate,
    TRUE AS IsActive
FROM silver_db.customers;
select * from gold_db.dim_customer LIMIT 10;

**Products dimension table**

In [0]:
%sql
CREATE OR REPLACE TABLE gold_db.dim_product AS
SELECT
    monotonically_increasing_id() AS ProductSK,
    ProductID,
    ProductName,
    Category,
    UnitPrice,
    CURRENT_DATE() AS EffectiveDate
FROM silver_db.products;
select * from gold_db.dim_product LIMIT 10;

**Dimension stores table**

In [0]:
%sql
CREATE OR REPLACE TABLE gold_db.dim_store AS
SELECT
    monotonically_increasing_id() AS StoreSK,
    StoreID,
    StoreName,
    Region
FROM silver_db.stores;
select * from gold_db.dim_store LIMIT 10;

**Fact table**

In [0]:
%sql
CREATE OR REPLACE TABLE gold_db.fact_sales AS
SELECT
    monotonically_increasing_id() AS SalesSK,
    s.TransactionID,
    c.CustomerSK,
    p.ProductSK,
    st.StoreSK,
    CAST(s.Quantity AS INT) AS Quantity,
    CAST(s.Quantity AS INT) * CAST(p.UnitPrice AS DOUBLE) AS Amount,
    s.TxnDate
FROM silver_db.sales s
INNER JOIN gold_db.dim_customer c
    ON TRIM(s.CustomerID) = TRIM(c.CustomerID) AND c.IsActive = TRUE
INNER JOIN gold_db.dim_product p
    ON TRIM(s.ProductID) = TRIM(p.ProductID)
INNER JOIN gold_db.dim_store st
    ON TRIM(s.StoreID) = TRIM(st.StoreID);